In [56]:
from datetime import datetime, time
import pandas as pd

# 1. Define file paths
INPUT_CSV = "CSE_16_spring_2026.csv"  # Replace with your actual file path
MON_OUTPUT_CSV = "attendance_monday_section.csv"
THURS_OUTPUT_CSV = "attendance_thursday_section.csv"

# 2. Load and clean the dataset
try:
    df = pd.read_csv(INPUT_CSV)
    print("CSV loaded successfully!")
except FileNotFoundError:
    print(f"Error: The file '{INPUT_CSV}' was not found.")

# Clean up column names (stripping hidden whitespace)
df.columns = df.columns.str.strip()

# 3. Parse Datetime (Handles standard formats like 'MM/DD/YYYY HH:MM:SS')
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

# Extract components needed for filtering and deduplication
df["Day_of_Week"] = df["Timestamp"].dt.day_name()
df["Class_Date"] = df["Timestamp"].dt.date
df["Class_Time"] = df["Timestamp"].dt.time

# 4. Define Section Time Windows (with your pre-calculated margins included)
# Update these times if your margins need exact minute adjustments
mon_start = time(15, 50, 0)  # e.g., 3:50 PM (5-min margin before 3:55)
mon_end = time(17, 15, 0)  # e.g., 5:15 PM (5-min margin after 5:10)

thurs_start = time(19, 0, 0)  # e.g., 7:00 PM (5-min margin before 7:05)
thurs_end = time(20, 25, 0)  # e.g., 8:25 PM (5-min margin after 8:20)


# 5. Helper function to process each section individually
def process_section(data, day_name, start_t, end_t, output_path):
    # Filter for the specific day and time window
    section_df = data[
        (data["Day_of_Week"] == day_name)
        & (data["Class_Time"] >= start_t)
        & (data["Class_Time"] <= end_t)
    ].copy()

    # Drop duplicates for the same student (First + Last Name) on the same calendar date
    section_unique = section_df.drop_duplicates(
        subset=["last name", "first name", "Class_Date"]
    )

    # Group by name and count total unique days attended
    attendance_summary = (
        section_unique.groupby(["last name", "first name"])
        .size()
        .reset_index(name="Times Shown Up")
    )

    # Export to CSV
    attendance_summary.to_csv(output_path, index=False)
    print(f" Saved {day_name} section summary to: {output_path}")
    return attendance_summary


# 6. Run the processor for both sections
print("\nProcessing attendance...")
monday_summary = process_section(
    df, "Monday", mon_start, mon_end, MON_OUTPUT_CSV
)
thursday_summary = process_section(
    df, "Thursday", thurs_start, thurs_end, THURS_OUTPUT_CSV
)

CSV loaded successfully!

Processing attendance...
 Saved Monday section summary to: attendance_monday_section.csv
 Saved Thursday section summary to: attendance_thursday_section.csv


In [57]:
# import pandas as pd

# # 1. Set your filename here
# filename = INPUT_CSV

# # 2. Read the CSV
# df = pd.read_csv(filename)

# # 3. Clean spaces and lowercase all string values
# # This applies a function across every cell; if it's text, it strips and lowercases it.
# df = df.map(
#     lambda x: x.strip().lower() if isinstance(x, str) else x
# )

# # 4. Clean column headers too (just in case they have extra spaces/uppercase)
# df.columns = df.columns.str.strip().str.lower()

# # 5. Save back to the exact same file
# df.to_csv(filename, index=False)

# print(f"Successfully cleaned and updated {filename}!")

In [58]:
# =========================================================================
# NEW CELL: Identify and print left-out rows (Self-Contained)
# =========================================================================

# Safety Check: Ensure the datetime columns actually exist in df
if "Day_of_Week" not in df.columns:
    print(
        "Creating missing datetime columns ('Day_of_Week', 'Class_Date', 'Class_Time')..."
    )
    df["Timestamp"] = pd.to_datetime(df["Timestamp"])
    df["Day_of_Week"] = df["Timestamp"].dt.day_name()
    df["Class_Date"] = df["Timestamp"].dt.date
    df["Class_Time"] = df["Timestamp"].dt.time

# 1. Define the identical logic masks used for filtering the sections
monday_mask = (
    (df["Day_of_Week"] == "Monday")
    & (df["Class_Time"] >= mon_start)
    & (df["Class_Time"] <= mon_end)
)

thursday_mask = (
    (df["Day_of_Week"] == "Thursday")
    & (df["Class_Time"] >= thurs_start)
    & (df["Class_Time"] <= thurs_end)
)

# 2. Use the invert operator (~) to find rows that belong to NEITHER section
left_out_df = df[~(monday_mask | thursday_mask)]

# 3. Display the results
print(f"\n--- Left-Out Rows Analysis ---")
print(f"Total entries ignored/left out: {len(left_out_df)}\n")

if not left_out_df.empty:
    print("Preview of left-out rows:")
    # Using columns safely by checking what's available
    available_cols = [
        col
        for col in [
            "Last name",
            "First name",
            "Day_of_Week",
            "Class_Time",
            "Timestamp",
        ]
        if col in df.columns
    ]
    display(left_out_df[available_cols])
else:
    print("Success! Every single row fit perfectly into your section criteria.")


--- Left-Out Rows Analysis ---
Total entries ignored/left out: 38

Preview of left-out rows:


,Day_of_Week,Class_Time,Timestamp
0,NaN,NaT,NaT
1,NaN,NaT,NaT
97,NaN,NaT,NaT
140,NaN,NaT,NaT
167,NaN,NaT,NaT
190,Thursday,20:29:55,2026-05-07 20:29:55
191,Thursday,20:44:47,2026-05-07 20:44:47
192,Thursday,21:22:16,2026-05-07 21:22:16
193,Friday,01:35:50,2026-05-08 01:35:50
194,Friday,02:04:53,2026-05-08 02:04:53
